# Differential-expression volcano plots for VCM populations

This notebook contains the differential-expression analyses and volcano plots used for VCM
comparisons. The GitHub version preserves the original statistical and plotting logic while
clearing notebook outputs, consolidating imports, and documenting the major analytical blocks.

Repeated volcano-plot operations are described once per section rather than annotated between
every similar plotting chunk.


## Setup

Imports and dependencies used for differential-expression analysis and figure generation.


In [ ]:
import bbknn
import h5py
import warnings
import os
import spatialdata_plot
import logging
from pathlib import Path
from adjustText import adjust_text
from gprofiler.gprofiler import GProfiler
import textwrap
from itertools import combinations


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
import spatialdata as sd
import spatialdata_io as sio


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:

# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Set `sc.settings.figdir` for the following analysis.
sc.settings.figdir = "Figures_forpaper/snRNAseq/VCMs/supplement"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Compute and store `out_dir`.
out_dir = Path("Figures_forpaper/snRNAseq/VCMs/supplement")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


## Load processed data

Load the annotated expression data and supporting tables used for the VCM comparisons.


In [ ]:
# Load the processed AnnData object.
VCMs = sc.read_h5ad("VCMs_harmonybysample_regressedout_annotated.h5ad")


In [ ]:
# Load the processed AnnData object.
VCMs = sc.read_h5ad("VCMs_harmonybysample_regressedout_annotated.h5ad")


In [ ]:
# Run `sc.pl.umap` for this analysis step.
sc.pl.umap(VCMs, color = 'celltype')


In [ ]:
# Make an independent copy of the selected data.
VCMs = VCMs[VCMs.obs["celltype"].isin(["CM1", "CM2", "CM3 (stressed)", "CM4 (Ifgga4 high)", "CM5"])].copy()


In [ ]:
# Define the values used for `set1_9`.
set1_9 = [
    "#6C5C8D", "#88CCEE", "#97B1AB", "#44AA99", "#117733", "#999933",
    "#AA4499", "#7BB4C3", "#CC6677", '#FF987B'

]


In [ ]:
# Define the values used for `order`.
order = ["CM1", "CM2", "CM3 (stressed)", "CM4 (Ifgga4 high)", "CM5"]


In [ ]:
# Compute and store `VCMs.obs['celltype']`.
VCMs.obs["celltype"] = pd.Categorical(VCMs.obs["celltype"], ordered = True, categories=order)


In [ ]:
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi_save = 300, figsize = (6, 6))
# Run `sc.pl.umap` for this analysis step.
sc.pl.umap(VCMs, color = 'celltype', palette=set1_9, size = 13, save = "VCMs_umap_moreclusters.pdf")


## Differential-expression testing

Run the comparison-specific differential-expression tests. These statistics provide the effect
sizes and significance values used in the downstream volcano plots.


In [ ]:
# Run differential-expression testing for this comparison.
sc.tl.rank_genes_groups(VCMs, groupby="celltype", method = "wilcoxon")


In [ ]:
# Run differential-expression testing for this comparison.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "celltype", standard_scale="var", n_genes=5)


In [ ]:
# Extract differential-expression results into a DataFrame.
df = sc.get.rank_genes_groups_df(VCMs, group="CM4 (Ifgga4 high)")

# Inspect the resulting values.
df.head()


In [ ]:
# Run differential-expression testing for this comparison.
sc.tl.rank_genes_groups(
    VCMs,
    groupby="celltype",
    groups=["CM4 (Ifgga4 high)"],
    reference="CM1",
    method="wilcoxon"
)


In [ ]:
# Extract differential-expression results into a DataFrame.
df = sc.get.rank_genes_groups_df(VCMs, group="CM4 (Ifgga4 high)")


## Prepare differential-expression results

Prepare gene-level fold changes and significance values and apply the thresholds used for plotting.
The original threshold and gene-selection choices are retained.


In [ ]:
# Filter the data to the observations/genes used in this comparison.
# Transform p-values to -log10 scale for volcano plotting.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(10, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs CM1")

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Display the completed figure.
plt.show()


## Volcano plots

Visualize differential-expression effect size versus statistical significance. Genes meeting the
comparison-specific criteria are highlighted and/or labelled according to the original figure code.
Related volcano plots below share this annotation.


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.

# Compute and store `df['pvals_adj_safe']`.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )

# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    arrowprops=dict(arrowstyle="-", linewidth=0.5)
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs CM1")


# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Save the finished figure to disk.
plt.savefig(out_dir/"DEGs_Wilcoxon_Ifgga4vsCM1_nolim.pdf", bbox_inches="tight")
# Display the completed figure.
plt.show()


## Selected gene labels

Annotate genes selected for emphasis in the manuscript figures. These selections and label-placement
choices are retained from the original plotting workflow.


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.

# Compute and store `df['pvals_adj_safe']`.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )

# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    only_move={"points": "y", "texts": "xy"},
    expand_text=(1.05, 1.1),
    expand_points=(1.05, 1.1),
    force_text=(0.1, 0.2),
    force_points=(0.1, 0.2),
    arrowprops=dict(
        arrowstyle="-",
        linewidth=0.3,
        alpha=0.5
    )
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs CM1")
# Run `plt.xlim` for this analysis step.
plt.xlim(-15, 15)

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Display the completed figure.
plt.show()


## Export figures

Save the resulting differential-expression/volcano plots for manuscript assembly.


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.

# Compute and store `df['pvals_adj_safe']`.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )

# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    only_move={"points": "y", "texts": "y"},
    arrowprops=dict(
        arrowstyle="-",
        linewidth=0.3,
        alpha=0.5
    )
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs CM1")
# Run `plt.xlim` for this analysis step.
plt.xlim(-15, 15)

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Save the finished figure to disk.
plt.savefig(out_dir/"DEGs_Wilcoxon_Ifgga4vsCM1.pdf", bbox_inches="tight")
# Display the completed figure.
plt.show()


In [ ]:
# Run differential-expression testing for this comparison.
sc.tl.rank_genes_groups(
    VCMs,
    groupby="celltype",
    groups=["CM4 (Ifgga4 high)"],
    reference="rest",
    method="wilcoxon"
)


In [ ]:
# Extract differential-expression results into a DataFrame.
df = sc.get.rank_genes_groups_df(
    VCMs,
    group="CM4 (Ifgga4 high)"
)


In [ ]:
# Filter the data to the observations/genes used in this comparison.
# Transform p-values to -log10 scale for volcano plotting.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)


In [ ]:
# Inspect the resulting values.
df.head()


In [ ]:
# Run `df.to_csv` for this analysis step.
df.to_csv("Ifgga4vsrest_snRNAseq_DEGs_wilcoxon.csv")


In [ ]:
# Save as Excel
df.to_excel(
    "Ifgga4vsrest_snRNAseq_DEGs_wilcoxon.xlsx",
    index=False
)


In [ ]:
# Select the required data and store it as `sig`.
sig= df.loc[df['significant']==True]


In [ ]:
# Make an independent copy of the selected data.
list = sig["names"].copy()


In [ ]:
# Compute and store `names_list`.
names_list = sig["names"].to_list()


In [ ]:
names_list


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.

# df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# df["significant"] = (
#     (df["pvals_adj"] < 0.05) &
#     (df["logfoldchanges"].abs() > 1)
#)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )


# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    arrowprops=dict(arrowstyle="-", linewidth=0.5)
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs rest")
# Run `plt.xlim` for this analysis step.
plt.xlim(-15, 15)

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Save the finished figure to disk.
plt.savefig(out_dir/"DEGs_Wilcoxon_Ifgga4vsrest.pdf", bbox_inches="tight")
# Display the completed figure.
plt.show()


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.

# Compute and store `df['pvals_adj_safe']`.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )


# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    arrowprops=dict(arrowstyle="-", linewidth=0.5)
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs rest")

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Save the finished figure to disk.
plt.savefig(out_dir/"DEGs_Wilcoxon_Ifgga4vsrest_nolim.pdf", bbox_inches="tight")
# Display the completed figure.
plt.show()


In [ ]:
# Run differential-expression testing for this comparison.
sc.tl.rank_genes_groups(
    VCMs,
    groupby="celltype",
    groups=["CM4 (Ifgga4 high)"],
    reference="CM3 (stressed)",
    method="wilcoxon"
)


In [ ]:
# Extract differential-expression results into a DataFrame.
df = sc.get.rank_genes_groups_df(VCMs, group="CM4 (Ifgga4 high)")


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.

# Compute and store `df['pvals_adj_safe']`.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )


# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    arrowprops=dict(arrowstyle="-", linewidth=0.5)
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs CM3 (stressed)")

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Save the finished figure to disk.
plt.savefig(out_dir/"DEGs_Wilcoxon_Ifgga4vsstressed_nolim.pdf", bbox_inches="tight")
# Display the completed figure.
plt.show()


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.

# Compute and store `df['pvals_adj_safe']`.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )


# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    arrowprops=dict(arrowstyle="-", linewidth=0.5)
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs CM3 (stressed)")

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Run `plt.xlim` for this analysis step.
plt.xlim(-15, 15)
# Save the finished figure to disk.
plt.savefig(out_dir/"DEGs_Wilcoxon_Ifgga4vsstressed_nolim.pdf", bbox_inches="tight")
# Display the completed figure.
plt.show()


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.
df["pvals_adj_safe"] = df["pvals_adj"].replace(0, 1e-300)
# Set `df['minus_log10_padj']` for the following analysis.
df["minus_log10_padj"] = -np.log10(df["pvals_adj_safe"])

# Calculate `df['significant']` from the existing values.
df["significant"] = (
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1)
)

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(8, 6))

# Plot the genes as points on the volcano plot.
plt.scatter(
    df["logfoldchanges"],
    df["minus_log10_padj"],
    c=df["significant"].map({True: "red", False: "grey"}),
    alpha=0.7,
    s=12
)

# Draw the fold-change threshold.
plt.axvline(1, linestyle="--", color="black", linewidth=1)
# Draw the fold-change threshold.
plt.axvline(-1, linestyle="--", color="black", linewidth=1)
# Draw the horizontal significance threshold.
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=1)

# label top significant genes
genes_to_label = (
    df[df["significant"]]
    .sort_values("pvals_adj")
    .head(20)
)

# Define the values used for `texts`.
texts = []
# Repeat the following operation for each item in the selected collection.
for _, row in genes_to_label.iterrows():
    texts.append(
        plt.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8
        )
    )


# Run `adjust_text` for this analysis step.
adjust_text(
    texts,
    arrowprops=dict(arrowstyle="-", linewidth=0.5)
)

# Label the x-axis.
plt.xlabel("log2 fold change")
# Label the y-axis.
plt.ylabel("-log10 adjusted p-value")
# Add the plot title.
plt.title("CM4 (Ifgga4 high) vs CM3 (stressed)")
# Run `plt.xlim` for this analysis step.
plt.xlim(-15, 15)

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Display the completed figure.
plt.show()


In [ ]:
# Run differential-expression testing for this comparison.
sc.tl.rank_genes_groups(
    VCMs,
    groupby="celltype",
    groups=["CM4 (Ifgga4 high)"],
    reference="rest",
    method="wilcoxon"
)


In [ ]:
# Extract differential-expression results into a DataFrame.
df = sc.get.rank_genes_groups_df(
    VCMs,
    group="CM4 (Ifgga4 high)"
)


In [ ]:
# Remove or identify missing values before downstream analysis.
# Use multiple-testing-adjusted significance values.
sig_deg = df.loc[
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"].abs() > 1),
    "names"
].dropna().unique().tolist()

# Compute and store `background_genes`.
background_genes = df["names"].dropna().unique().tolist()


In [ ]:
sig_deg


In [ ]:

# Compute and store `gp`.
gp = GProfiler(return_dataframe=True)

# Compute and store `go_res`.
go_res = gp.profile(
    organism="mmusculus",
    query=sig_deg,
    background=background_genes,
    sources=["GO:BP", "GO:MF", "GO:CC", "REAC", "KEGG", "WP"],
    no_evidences=False
)

# Inspect the resulting values.
go_res.head(20)


In [ ]:
# Compute and store `go_res`.
go_res = gp.profile(
    organism="mmusculus",
    query=sig_deg,
    background=background_genes,
    sources=["GO:BP"],
    no_evidences=False
)


In [ ]:
# Inspect the resulting values.
go_res.head()


In [ ]:
# Remove or identify missing values before downstream analysis.
# Use multiple-testing-adjusted significance values.
up_genes = df.loc[
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"] > 1),
    "names"
].dropna().unique().tolist()

# Compute and store `down_genes`.
down_genes = df.loc[
    (df["pvals_adj"] < 0.05) &
    (df["logfoldchanges"] < -1),
    "names"
].dropna().unique().tolist()

# Compute and store `background_genes`.
background_genes = df["names"].dropna().unique().tolist()


In [ ]:
# Compute and store `go_bp_up`.
go_bp_up = gp.profile(
    organism="mmusculus",
    query=up_genes,
    background=background_genes,
    sources=["GO:BP"],
    no_evidences=False
)

# go_bp_down = gp.profile(
#     organism="mmusculus",
#     query=down_genes,
#     background=background_genes,
#     sources=["GO:BP"]
# )


In [ ]:
# Inspect the resulting values.
go_bp_up.head(10)


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.
import matplotlib.pyplot as plt

# Make an independent copy of the selected data.
go_bp_plot = go_res.copy()

# sort by significance
go_bp_plot = go_bp_plot.sort_values("p_value")

# make plotting column
go_bp_plot["minus_log10_p"] = -np.log10(go_bp_plot["p_value"])


In [ ]:
# Work on an explicit copy of the selected data.
top_n = 15
# Make an independent copy of the selected data.
top_terms = go_bp_plot.head(top_n).copy()

# Run `plt.figure` for this analysis step.
plt.figure(figsize=(10, 5))

# Run `plt.barh` for this analysis step.
plt.barh(
    top_terms["name"],
    top_terms["minus_log10_p"]
)

# Run `plt.gca().invert_yaxis` for this analysis step.
plt.gca().invert_yaxis()
# Label the x-axis.
plt.xlabel("-log10 p-value")
# Label the y-axis.
plt.ylabel("")
# Add the plot title.
plt.title("Top GO Biological Process terms")

# Adjust spacing to prevent overlapping figure elements.
plt.tight_layout()
# Display the completed figure.
plt.show()


In [ ]:
# Run `top_terms.to_csv` for this analysis step.
top_terms.to_csv(out_dir/"GOBP_Ifgga4vsrest.csv")


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define `plot_gobp_dotplot()` for reuse in the analysis below.
def plot_gobp_dotplot(
    go_df,
    query_size,
    title,
    top_n=20,
    wrap_width=35,
    dot_scale=18,
    figsize=(8, 8),
    save_path=None
):
    # Prepare data
    plot_df = go_df.copy()
    plot_df = plot_df.sort_values("p_value", ascending=True).head(top_n).copy()

    plot_df["GeneRatio"] = plot_df["intersection_size"] / query_size
    plot_df["Count"] = plot_df["intersection_size"]
    plot_df["p.adjust"] = plot_df["p_value"]

    # Sort by GeneRatio so dots form a smoother line
    plot_df = plot_df.sort_values("GeneRatio", ascending=True).copy()

    plot_df["name_wrapped"] = plot_df["name"].apply(
        lambda x: "\n".join(textwrap.wrap(x, width=wrap_width))
    )

    plot_df["y"] = np.arange(len(plot_df))

    # Plot
    fig, ax = plt.subplots(figsize=figsize)

    scatter = ax.scatter(
        plot_df["GeneRatio"],
        plot_df["y"],
        s=plot_df["Count"] * dot_scale,
        c=plot_df["p.adjust"],
        cmap="coolwarm_r",
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85
    )

    ax.set_yticks(plot_df["y"])
    ax.set_yticklabels(plot_df["name_wrapped"])

    ax.set_xlabel("Gene Ratio")
    ax.set_ylabel("")
    ax.set_title(title)

    ax.grid(True, axis="both", linestyle="-", linewidth=0.5, alpha=0.3)
    ax.set_axisbelow(True)

    # Add some x-axis padding
    xmax = plot_df["GeneRatio"].max()
    ax.set_xlim(0, xmax * 1.15)

    # Colorbar
    cbar = plt.colorbar(scatter, ax=ax, pad=0.03)
    cbar.set_label("p.adjust")

    # Size legend
    legend_counts = np.linspace(
        plot_df["Count"].min(),
        plot_df["Count"].max(),
        4
    ).astype(int)

    legend_counts = sorted(set(legend_counts))

    for count in legend_counts:
        ax.scatter(
            [],
            [],
            s=count * dot_scale,
            c="white",
            edgecolor="black",
            linewidth=0.5,
            label=str(count)
        )

    ax.legend(
        title="Genes overlap",
        bbox_to_anchor=(1.28, 0.45),
        loc="center left",
        frameon=False
    )

    plt.tight_layout(rect=[0, 0, 0.82, 1])

    plt.savefig(save_path, bbox_inches="tight")

    plt.show()

    return plot_df


In [ ]:
# Compute and store `up_plot_df`.
up_plot_df = plot_gobp_dotplot(
    go_df=go_bp_up,
    query_size=len(up_genes),
    title="GO Biological Process enrichment: upregulated DEGs",
    top_n=20,
    figsize=(15, 10),
    save_path=out_dir / "GO_BP_upregulated_DEGs_Ifgga4vsrestwilcoxon.pdf"
)


In [ ]:
# Build a DataFrame for downstream analysis/plotting.
# Sort the results for easier inspection or plotting.
import pandas as pd
import numpy as np

# Define `parse_gprofiler_genes()` for reuse in the analysis below.
def parse_gprofiler_genes(x):
    if x is None:
        return set()
    
    if isinstance(x, float) and pd.isna(x):
        return set()
    
    if isinstance(x, list):
        genes = []
        for item in x:
            if isinstance(item, list):
                genes.extend(item)
            else:
                genes.append(item)
        return set(map(str, genes))
    
    if isinstance(x, str):
        for sep in [",", ";", "/"]:
            if sep in x:
                return set(g.strip() for g in x.split(sep) if g.strip())
        return set(x.split())
    
    return set()


# Define `make_go_overlap_table()` for reuse in the analysis below.
def make_go_overlap_table(
    go_df,
    gene_col=None,
    top_n=50,
    min_jaccard=0.3,
    min_overlap_coef=0.6
):
    """
    Creates a table of GO-term pairs with overlapping enriched genes.
    Useful for manual redundancy inspection.
    """

    go_df = go_df.copy()

    if gene_col is None:
        possible_gene_cols = [
            "intersections",
            "intersection",
            "intersecting_genes",
            "genes"
        ]
        gene_col = next((c for c in possible_gene_cols if c in go_df.columns), None)

    if gene_col is None:
        raise ValueError(
            "No gene intersection column found. "
            "Check go_bp_up.columns.tolist(). "
            "You probably need no_evidences=False in gp.profile()."
        )

    go_df = go_df.sort_values("p_value").head(top_n).copy()
    go_df["gene_set"] = go_df[gene_col].apply(parse_gprofiler_genes)

    rows = []

    for i, j in combinations(go_df.index, 2):
        row1 = go_df.loc[i]
        row2 = go_df.loc[j]

        genes1 = row1["gene_set"]
        genes2 = row2["gene_set"]

        shared_genes = genes1 & genes2
        union_genes = genes1 | genes2

        if len(shared_genes) == 0:
            continue

        jaccard = len(shared_genes) / len(union_genes)
        overlap_coef = len(shared_genes) / min(len(genes1), len(genes2))

        if (jaccard >= min_jaccard) or (overlap_coef >= min_overlap_coef):
            rows.append({
                "term_1": row1["name"],
                "GO_ID_1": row1["native"],
                "p_value_1": row1["p_value"],
                "count_1": row1["intersection_size"],

                "term_2": row2["name"],
                "GO_ID_2": row2["native"],
                "p_value_2": row2["p_value"],
                "count_2": row2["intersection_size"],

                "shared_gene_count": len(shared_genes),
                "jaccard": jaccard,
                "overlap_coef": overlap_coef,
                "shared_genes": ", ".join(sorted(shared_genes))
            })

    overlap_df = pd.DataFrame(rows)

    if len(overlap_df) > 0:
        overlap_df = overlap_df.sort_values(
            ["overlap_coef", "jaccard", "shared_gene_count"],
            ascending=False
        )

    return overlap_df


In [ ]:
# Compute and store `go_overlap_up`.
go_overlap_up = make_go_overlap_table(
    go_bp_up,
    top_n=50,
    min_jaccard=0.3,
    min_overlap_coef=0.6
)

# Inspect the resulting values.
go_overlap_up.head(30)


In [ ]:
# Build a DataFrame for downstream analysis/plotting.
# Sort the results for easier inspection or plotting.
import pandas as pd
import numpy as np

# Define `parse_gprofiler_genes()` for reuse in the analysis below.
def parse_gprofiler_genes(x):
    if x is None:
        return set()
    
    if isinstance(x, float) and pd.isna(x):
        return set()
    
    if isinstance(x, list):
        genes = []
        for item in x:
            if isinstance(item, list):
                genes.extend(item)
            else:
                genes.append(item)
        return set(map(str, genes))
    
    if isinstance(x, str):
        for sep in [",", ";", "/"]:
            if sep in x:
                return set(g.strip() for g in x.split(sep) if g.strip())
        return set(x.split())
    
    return set()


# Define `make_full_overlap_groups()` for reuse in the analysis below.
def make_full_overlap_groups(
    go_df,
    gene_col=None,
    top_n=50,
    mode="contained",  # "contained" or "exact"
):
    """
    Groups GO terms that fully overlap.

    mode="exact":
        terms must have exactly the same enriched genes.

    mode="contained":
        one term's enriched genes can be fully contained within another term.
        This is usually more useful for GO redundancy.
    """

    go_df = go_df.copy()

    if gene_col is None:
        possible_gene_cols = [
            "intersections",
            "intersection",
            "intersecting_genes",
            "genes"
        ]
        gene_col = next((c for c in possible_gene_cols if c in go_df.columns), None)

    if gene_col is None:
        raise ValueError(
            "No gene intersection column found. "
            "Check go_bp_up.columns.tolist()."
        )

    go_df = go_df.sort_values("p_value").head(top_n).copy()
    go_df["gene_set"] = go_df[gene_col].apply(parse_gprofiler_genes)
    go_df = go_df[go_df["gene_set"].apply(len) > 0].copy()

    indices = list(go_df.index)

    # union-find helper
    parent = {i: i for i in indices}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        root_x = find(x)
        root_y = find(y)
        if root_x != root_y:
            parent[root_y] = root_x

    # connect fully overlapping terms
    for i, j in combinations(indices, 2):
        genes_i = go_df.loc[i, "gene_set"]
        genes_j = go_df.loc[j, "gene_set"]

        shared = len(genes_i & genes_j)
        union_size = len(genes_i | genes_j)
        smaller = min(len(genes_i), len(genes_j))

        jaccard = shared / union_size if union_size > 0 else 0
        overlap_coef = shared / smaller if smaller > 0 else 0

        if mode == "exact":
            full_overlap = np.isclose(jaccard, 1.0)
        elif mode == "contained":
            full_overlap = np.isclose(overlap_coef, 1.0)
        else:
            raise ValueError("mode must be 'exact' or 'contained'")

        if full_overlap:
            union(i, j)

    # assign groups
    go_df["overlap_group"] = [find(i) for i in go_df.index]

    grouped_rows = []

    for group_id, sub in go_df.groupby("overlap_group"):
        sub = sub.sort_values("p_value")

        representative = sub.iloc[0]

        # genes shared by all terms in the group
        shared_by_all = set.intersection(*sub["gene_set"].tolist())

        # all genes appearing in the group
        union_genes = set.union(*sub["gene_set"].tolist())

        grouped_rows.append({
            "representative_term": representative["name"],
            "representative_GO_ID": representative["native"],
            "representative_p_value": representative["p_value"],
            "n_terms_in_group": len(sub),

            "all_terms": " | ".join(sub["name"].tolist()),
            "all_GO_IDs": " | ".join(sub["native"].tolist()),
            "all_p_values": " | ".join(sub["p_value"].astype(str).tolist()),

            "min_intersection_size": sub["intersection_size"].min(),
            "max_intersection_size": sub["intersection_size"].max(),

            "genes_shared_by_all": ", ".join(sorted(shared_by_all)),
            "all_genes_in_group": ", ".join(sorted(union_genes))
        })

    grouped_df = pd.DataFrame(grouped_rows)

    grouped_df = grouped_df.sort_values(
        ["representative_p_value", "n_terms_in_group"],
        ascending=[True, False]
    )

    return grouped_df


In [ ]:
# Run `pd.set_option` for this analysis step.
pd.set_option("display.max_colwidth", None)
# Run `pd.set_option` for this analysis step.
pd.set_option("display.max_rows", 100)
# Run `pd.set_option` for this analysis step.
pd.set_option("display.width", 200)


In [ ]:
# Compute and store `go_exact_overlap_groups`.
go_exact_overlap_groups = make_full_overlap_groups(
    go_bp_up,
    top_n=50,
    mode="exact"
)
# Inspect the resulting values.
go_exact_overlap_groups.head(20)


In [ ]:
# Sort the results for easier inspection or plotting.
# Filter the data to the observations/genes used in this comparison.
# 1. Select the GO terms you want to keep
manual_keep_go_ids = [
    "GO:0045087",  # innate immune response
    "GO:0002376",  # immune system process
    "GO:0001562",  # response to protozoan
    "GO:0034341",  # response to type II interferon
    "GO:0034097",  # response to cytokine
    "GO:0009617",  # response to bacterium
    #"GO:0042742",  # defense response to bacterium
    #"GO:0045088",  # regulation of innate immune response
    #"GO:0050776",  # regulation of immune response
    "GO:0051607",  # defense response to virus
]

# Make an independent copy of the selected data.
go_bp_up_manual_selected = go_bp_up.loc[
    go_bp_up["native"].isin(manual_keep_go_ids)
].copy()

# 2. Add plotting columns
go_bp_up_manual_selected["GeneRatio"] = (
    go_bp_up_manual_selected["intersection_size"] / len(up_genes)
)

# Select the required data and store it as `go_bp_up_manual_selected['Count']`.
go_bp_up_manual_selected["Count"] = go_bp_up_manual_selected["intersection_size"]
# Select the required data and store it as `go_bp_up_manual_selected['adjusted_p_value']`.
go_bp_up_manual_selected["adjusted_p_value"] = go_bp_up_manual_selected["p_value"]

# 3. Order by overlap and significance
# This means: larger gene overlap first, and within similar overlap, stronger p-value first
go_bp_up_manual_selected = go_bp_up_manual_selected.sort_values(
    ["GeneRatio", "p_value"],
    ascending=[False, True]
).copy()

# Run `display` for this analysis step.
display(
    go_bp_up_manual_selected[
        ["native", "name", "GeneRatio", "Count", "p_value"]
    ]
)


In [ ]:
# Filter the data to the observations/genes used in this comparison.
# Work on an explicit copy of the selected data.
import numpy as np
import matplotlib.pyplot as plt

# Define `plot_gobp_dotplot_selected_ordered()` for reuse in the analysis below.
def plot_gobp_dotplot_selected_ordered(
    go_df,
    title,
    wrap_width=35,
    dot_scale=18,
    figsize=(15, 8),
    save_path=None
):
    plot_df = go_df.copy()

    plot_df["name_wrapped"] = plot_df["name"].apply(
        lambda x: "\n".join(textwrap.wrap(x, width=wrap_width))
    )

    # Reverse only visually so the first dataframe row appears at the top
    plot_df = plot_df.iloc[::-1].copy()
    plot_df["y"] = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=figsize)

    scatter = ax.scatter(
        plot_df["GeneRatio"],
        plot_df["y"],
        s=plot_df["Count"] * dot_scale,
        c=plot_df["adjusted_p_value"],
        cmap="coolwarm_r",
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85
    )

    ax.set_yticks(plot_df["y"])
    ax.set_yticklabels(plot_df["name_wrapped"])

    ax.set_xlabel("Gene Ratio")
    ax.set_ylabel("")
    ax.set_title(title)

    ax.grid(True, axis="both", linestyle="-", linewidth=0.5, alpha=0.3)
    ax.set_axisbelow(True)

    xmax = plot_df["GeneRatio"].max()
    ax.set_xlim(0, xmax * 1.15)

    cbar = plt.colorbar(scatter, ax=ax, pad=0.03)
    cbar.set_label("g:Profiler adjusted p-value")

    legend_counts = np.linspace(
        plot_df["Count"].min(),
        plot_df["Count"].max(),
        4
    ).astype(int)

    legend_counts = sorted(set(legend_counts))

    for count in legend_counts:
        ax.scatter(
            [],
            [],
            s=count * dot_scale,
            c="white",
            edgecolor="black",
            linewidth=0.5,
            label=str(count)
        )

    ax.legend(
        title="Genes overlap",
        bbox_to_anchor=(1.28, 0.45),
        loc="center left",
        frameon=False
    )

    plt.tight_layout(rect=[0, 0, 0.82, 1])

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")

    plt.show()

    return plot_df


In [ ]:
# Compute and store `up_plot_df`.
up_plot_df = plot_gobp_dotplot_selected_ordered(
    go_df=go_bp_up_manual_selected,
    title="GO Biological Process enrichment: upregulated DEGs",
    figsize=(13, 6),
    save_path=out_dir / "GO_BP_upregulated_DEGs_manual_selected_overlap_significance_order.pdf"
)
